## PIPELINE (CatBoost + Pseudo Labels + Multi-Seed Ensemble)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier

# ============================================================
# Load Data
# ============================================================
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e7/test.csv")

X = train.drop(columns=["id", "health_condition"]).copy()
X_test = test.drop(columns=["id"]).copy()
y = train["health_condition"]

base_cat_cols = [
    'diet_type', 'stress_level', 'sleep_quality', 
    'physical_activity_level', 'smoking_alcohol', 'gender'
]

# Fill missing values BEFORE feature engineering to prevent "nan" strings
X[base_cat_cols] = X[base_cat_cols].fillna("Missing").astype(str)
X_test[base_cat_cols] = X_test[base_cat_cols].fillna("Missing").astype(str)

# ============================================================
# Feature Engineering
# ============================================================
def apply_feature_engineering(df):
    # Handle missing numerical BMI values before binning
    df["bmi"] = df["bmi"].fillna(df["bmi"].median())
    
    # BMI class
    df["bmi_class"] = pd.cut(
        df["bmi"], bins=[0, 18.5, 25, 30, np.inf], 
        labels=["Under", "Normal", "Over", "Obese"], include_lowest=True
    ).astype(str)
    
    # Categorical Interactions
    df["bmi_activity"] = df["bmi_class"] + "_" + df["physical_activity_level"].astype(str)
    df["bmi_diet"] = df["bmi_class"] + "_" + df["diet_type"].astype(str)
    df["diet_smoking"] = df["diet_type"].astype(str) + "_" + df["smoking_alcohol"].astype(str)
    df["gender_bmi"] = df["gender"].astype(str) + "_" + df["bmi_class"]
    df["sleep_activity"] = df["sleep_quality"].astype(str) + "_" + df["physical_activity_level"].astype(str)
    return df

X = apply_feature_engineering(X)
X_test = apply_feature_engineering(X_test)

all_cat_cols = base_cat_cols + [
    'bmi_class', 'bmi_activity', 'bmi_diet', 
    'diet_smoking', 'gender_bmi', 'sleep_activity'
]

# Force categorical columns to string type for CatBoost
for c in all_cat_cols:
    X[c] = X[c].astype(str)
    X_test[c] = X_test[c].astype(str)

# Target Encoding
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Pass column names directly to CatBoost
cat_features = all_cat_cols 

# ============================================================
# Model Parameters (FIXED)
# ============================================================
cb_params = {
    "iterations": 500,
    "learning_rate": 0.025,
    "depth": 8,
    "l2_leaf_reg": 5,
    "random_strength": 2,
    "bagging_temperature": 1,  
    "loss_function": "MultiClass",  
    "eval_metric": "MultiClass",    
    "bootstrap_type": "Bayesian",
    "auto_class_weights": "Balanced",
    "task_type": "CPU",
    "verbose": 0
}

def train_model(X_train, y_train, X_val, y_val, seed):
    model = CatBoostClassifier(**cb_params, random_seed=seed)
    model.fit(
        X_train, y_train, 
        cat_features=cat_features, 
        eval_set=(X_val, y_val), 
        early_stopping_rounds=100, 
        use_best_model=True
    )
    return model

# ============================================================
# Main Training Loop
# ============================================================
seeds = [42, 99]
final_test_probs = np.zeros((len(X_test), len(le.classes_)))

for seed in seeds:
    print(f"Training Seed {seed}...")
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    
    for tr_idx, val_idx in skf.split(X, y_enc):
        model = train_model(X.iloc[tr_idx], y_enc[tr_idx], X.iloc[val_idx], y_enc[val_idx], seed)
        final_test_probs += model.predict_proba(X_test) / (len(seeds) * skf.n_splits)

# ============================================================
# High-Confidence Pseudo-Labeling & Final Refit
# ============================================================
print("Applying Pseudo-labels...")
confidence = np.max(final_test_probs, axis=1)
high_conf = confidence > 0.975 

X_final = pd.concat([X, X_test[high_conf]], ignore_index=True)
y_final = np.concatenate([y_enc, np.argmax(final_test_probs[high_conf], axis=1)])

final_params = cb_params.copy()
final_params["iterations"] = 1200

print(f"Refitting final model on {len(X_final)} rows...")
final_model = CatBoostClassifier(**final_params, random_seed=42)
final_model.fit(X_final, y_final, cat_features=cat_features)

# ============================================================
# Save Submission
# ============================================================
submission = pd.DataFrame({
    "id": test["id"],
    "health_condition": le.inverse_transform(np.argmax(final_model.predict_proba(X_test), axis=1))
})
submission.to_csv("submission.csv", index=False)
print("Submission saved successfully!")

print(submission.head(20))

Training Seed 42...
Training Seed 99...
Applying Pseudo-labels...
Refitting final model on 719764 rows...
Submission saved successfully!
        id health_condition
0   690088        unhealthy
1   690089        unhealthy
2   690090          at-risk
3   690091          at-risk
4   690092        unhealthy
5   690093              fit
6   690094          at-risk
7   690095          at-risk
8   690096          at-risk
9   690097          at-risk
10  690098        unhealthy
11  690099          at-risk
12  690100          at-risk
13  690101          at-risk
14  690102              fit
15  690103          at-risk
16  690104        unhealthy
17  690105          at-risk
18  690106          at-risk
19  690107          at-risk
